In [ ]:
from binder_session import launch_binder_and_get_jupyter_url, BinderTerminal, shutdown_session

binder_base_url = "https://binder.intel4coro.de/build/gh"
repo_path = "yxzhan/robot-lobby/main"

launch_url = f"{binder_base_url}/{repo_path}"

ok, result = launch_binder_and_get_jupyter_url(launch_url, verbose=True)

In [ ]:
f"{result['url']}robot-lobby/?token={result['token']}"

In [ ]:
term = BinderTerminal(result["url"], result["token"])

In [ ]:
# out, rc = term.run("ls")
# print(f"[exit {rc}]\n{out}")

out, rc = term.run("ls", stream=True, timeout=60)

## Sync the model library (mjcf_assets/) into the session

`sync_assets.py` compares the local `mjcf_assets/` with the session's (`public/mjcf_assets` there, the shared `/mnt/dev-tools/mjcf_assets`) and sends only what is new or changed: packed into tars of at most 512 MB, uploaded through the Jupyter Contents API (as JupyterLab uploads files), unpacked there through the terminal, then the library is indexed again for the Assets panel.

- Files of the same size whose only difference is the mtime are compared by md5 and left alone if the content is the same.
- Files **newer there** (changed by someone else) are kept unless `overwrite_newer=True`.
- Files **only there** are kept unless `delete=True` (the volume is shared).
- `saved_stage/` and `.git` directories are never compared or sent.
- An interrupted sync just runs again: what arrived already compares equal.

In [ ]:
from sync_assets import sync_assets

# Dry run: lists what would change, writes nothing.
plan = sync_assets(term)

In [ ]:
# The sync itself: sends the new and changed files, then indexes the library again.
plan = sync_assets(term, dry_run=False)

# Also overwrite files that are newer there, or delete files only there:
# plan = sync_assets(term, dry_run=False, overwrite_newer=True, delete=True)

In [ ]:
# What the plan holds, e.g. the files only there or newer there:
print(plan['remote_only'])
print(plan['remote_newer'])

In [ ]:
term.close()

In [ ]:
shutdown_session(result["url"], result["token"])